In [0]:
# --- DATA QUALITY & INTEGRITY TESTS ---
# Description: Validating data across Bronze, Silver, and Gold layers.

from pyspark.sql.functions import col, count, when, isnull

# Load tables (metadata)
bronze_df = spark.table("main.market_project.bronze_bank_prices")
silver_df = spark.table("main.market_project.silver_bank_prices")
gold_df   = spark.table("main.market_project.gold_bank_metrics")

In [0]:
# --- TEST 1: SILVER Schema Consistency ---
print("TEST 1: Validating Silver Schema...")

expected_columns = ['event_timestamp', 'open_price', 'high_price', 'low_price', 'close_price', 'volume', 'bank_name']
actual_columns = silver_df.columns

if set(expected_columns).issubset(set(actual_columns)):
    print("✅ Success: Silver schema is correct.")
else:
    missing = set(expected_columns) - set(actual_columns)
    print(f"❌ Error: Schema mismatch! Missing: {missing}")

In [0]:
# --- TEST 2: SILVER Data Completeness (Null Checks) ---
print("TEST 2: Checking for NULL values in Silver layer...")

null_report = silver_df.select([count(when(isnull(c), c)).alias(c) for c in silver_df.columns])
display(null_report)

In [0]:
# --- TEST 3: SILVER Row Count Lineage ---
print("TEST 3: Row count comparison & Data Loss Analysis...")

b_count = bronze_df.count()
s_count = silver_df.count()

if b_count > 0:
    loss_pct = ((b_count - s_count) / b_count) * 100
    print(f"Bronze Rows: {b_count}")
    print(f"Silver Rows: {s_count}")
    print(f"Data Loss: {loss_pct:.2f}%")
    
    if loss_pct > 10:
        print(f"🚨 ALERT: High data loss detected ({loss_pct:.2f}%)!")
        
        # Identification of dropped records
        dropped_rows_sample = bronze_df.join(
            silver_df, 
            bronze_df.Datetime == silver_df.event_timestamp, 
            "left_anti"
        ).limit(20)
        
        print("Sample of dropped records from Bronze:")
        display(dropped_rows_sample)
    else:
        print("✅ Success: Data loss is within acceptable limits.")
else:
    print("❌ Error: Bronze table is empty!")

In [0]:
# --- TEST 4: SILVER Visual Sanity Check ---
print("TEST 4: Visual sample of Silver metrics...")
display(silver_df.limit(10))

In [0]:
# --- TEST 5: GOLD Row Count & Business Logic ---
print("\nTEST 5: GOLD Row Count & Logic Check...")

g_count = gold_df.count()
print(f"Gold Rows: {g_count}")

if g_count == 0:
    print("❌ Error: Gold table is empty!")
elif g_count > s_count:
    print("⚠️ Warning: Gold row count is higher than Silver. Check for duplicates!")
else:
    print("✅ Success: Gold table has data.")

invalid_prices = gold_df.filter(col("close_price") <= 0).count()
if invalid_prices > 0:
    print(f"❌ Error: Found {invalid_prices} rows with invalid (negative/zero) prices!")
else:
    print("✅ Success: All prices in Gold are valid.")

In [0]:
# --- TEST 6: GOLD Visual Sample ---
print("\nTEST 6: Visual sample of GOLD business metrics...")
display(gold_df.select("bank_name", "event_timestamp", "daily_return_pct", "moving_avg_7d").limit(10))